In [ ]:
from .Connect import get_spark 
from pyspark.sql.functions import col

In [32]:
import os

In [34]:
path = os.getcwd()
HADOOP_HOME = r"C:\hadoop"
print(path)

c:\Users\Si3ma\Desktop\spark_simulation\open_sky_pipeline\Notebooks


In [2]:
spark=get_spark()

In [83]:
df=spark.read.format('delta').load("s3a://silver/aircraft").where((col('isPoland')==True) & (col('position_source_name')=='ADS-B'))
df_hist=spark.read.format('delta').load("s3a://silver/aircraft_hist").where((col('isPoland')==True) & (col('position_source_name')=='ADS-B'))
df.show()
df_hist.show()

+------+--------+--------------------+-------------+------------+---------+--------+------------+---------+--------+----------+-------------+-------+-------------+------+-----+---------------+--------+--------------------+-------------+-------------------+-------------------+-----------------+-----------------+--------------------+--------+
|icao24|callsign|      origin_country|time_position|last_contact|longitude|latitude|geo_altitude|on_ground|velocity|true_track|vertical_rate|sensors|baro_altitude|squawk|  spi|position_source|category| ingestion_timestamp|altitude_diff|     last_contact_h|    time_position_h|vertical_category|aircraft_category|position_source_name|isPoland|
+------+--------+--------------------+-------------+------------+---------+--------+------------+---------+--------+----------+-------------+-------+-------------+------+-----+---------------+--------+--------------------+-------------+-------------------+-------------------+-----------------+-----------------+--

In [84]:
df.select('callsign','last_contact_h','time_position_h','longitude','latitude','geo_altitude','baro_altitude','altitude_diff','true_track','velocity','vertical_category').show()

+--------+-------------------+-------------------+---------+--------+------------+-------------+-------------+----------+--------+-----------------+
|callsign|     last_contact_h|    time_position_h|longitude|latitude|geo_altitude|baro_altitude|altitude_diff|true_track|velocity|vertical_category|
+--------+-------------------+-------------------+---------+--------+------------+-------------+-------------+----------+--------+-----------------+
|CCA962  |2026-08-19 14:41:25|2026-08-19 14:25:14|  19.0346| 53.9926|    10256.52|      10058.4|    198.11914|     38.37|   271.0|Constant Altitude|
|SPKIC   |2026-08-19 14:41:29|2026-08-19 14:41:29|  18.6425| 50.2475|       685.8|       624.84|     60.95996|    173.23|   30.57|       Descending|
|SPKOI   |2026-08-19 14:41:35|2026-08-19 14:41:33|  21.4071| 50.2314|      784.86|       792.48|    -7.619995|    267.56|    24.2|       Descending|
|KLC42X  |2026-08-19 14:41:29|2026-08-19 14:41:28|  14.8287| 52.5193|     8473.44|       8229.6|    243.84

In [5]:
df.select('icao24','origin_country','on_ground','squawk', 'spi','aircraft_category').show()

+------+--------------------+---------+------+-----+-----------------+
|icao24|      origin_country|on_ground|squawk|  spi|aircraft_category|
+------+--------------------+---------+------+-----+-----------------+
|78115f|               China|    false|  6411|false|            Heavy|
|48a502|              Poland|    false|  NULL|false|          No Info|
|48a5c8|              Poland|    false|  0002|false|          No Info|
|4855d1|Kingdom of the Ne...|    false|  1000|false|          No Info|
|45d066|             Denmark|    false|  1000|false|            Large|
|4780c7|              Norway|    false|  1000|false|          No Info|
|48af19|              Poland|    false|  6574|false|          No Info|
|48af01|              Poland|    false|  1000|false|          No Info|
|781eb8|               China|    false|  2305|false|          No Info|
|780c58|               China|    false|  7637|false|          No Info|
|47945c|              Norway|    false|  4264|false|          No Info|
|48191

In [106]:
df_hist.where(col('on_ground')==False).groupBy(col('ingestion_timestamp')).avg('velocity','baro_altitude').orderBy(col('ingestion_timestamp'),ascending=False).select('ingestion_timestamp',col('avg(velocity)').alias('avg_velocity'),col('avg(baro_altitude)').alias('avg_baro_alt')).show()

+--------------------+------------------+-----------------+
| ingestion_timestamp|      avg_velocity|     avg_baro_alt|
+--------------------+------------------+-----------------+
|2026-08-19 14:41:...|155.72843754291534|5956.141229947408|
|2026-08-17 22:27:...| 234.1506905788663|8236.957219967897|
+--------------------+------------------+-----------------+



In [130]:
from pyspark.sql.functions import avg,count

df_all=df_hist.groupBy(col('ingestion_timestamp')).count().orderBy(col('count'),ascending=False).select('ingestion_timestamp',col('count').alias('all_observation_count'))

df_ground=df_hist.where(col('on_ground')==True ).groupBy(col('ingestion_timestamp')).count().orderBy([col('ingestion_timestamp')],ascending=False).select('ingestion_timestamp',col('count').alias('on_ground_count'))

df_velocity=df_hist.where(col('on_ground')==False).groupBy(col('ingestion_timestamp')).agg(
    avg('velocity').alias('avg_velocity'),
    avg('baro_altitude').alias('avg_baro_alt'),
    count('*').alias('count_flying')
    ).orderBy(col('ingestion_timestamp'),ascending=False).select('ingestion_timestamp','avg_velocity','avg_baro_alt','count_flying')

df_all=df_all.alias('da')\
.join(df_ground.alias('dg'),on=["ingestion_timestamp"],how="left")\
.join(df_velocity.alias('dv'),on=["ingestion_timestamp"],how="left")\
.select('ingestion_timestamp','all_observation_count','count_flying','on_ground_count','dv.avg_velocity','dv.avg_baro_alt').show()


df_cat=df_hist.where(col('on_ground')==False)\
.groupBy(col('ingestion_timestamp'),col('vertical_category'))\
.agg(
    avg('baro_altitude').alias('avg_baro_alt'),
    avg('velocity').alias('avg_velocity'),
    count('*').alias('flying_count')
    )\
.orderBy([col('ingestion_timestamp'),col('vertical_category')],ascending=False)

df_cat.show()


+--------------------+---------------------+------------+---------------+------------------+-----------------+
| ingestion_timestamp|all_observation_count|count_flying|on_ground_count|      avg_velocity|     avg_baro_alt|
+--------------------+---------------------+------------+---------------+------------------+-----------------+
|2026-08-17 22:27:...|                  352|         348|              4| 234.1506905788663|8236.957219967897|
|2026-08-19 14:41:...|                  100|          96|              4|155.72843754291534|5956.141229947408|
+--------------------+---------------------+------------+---------------+------------------+-----------------+

+--------------------+-----------------+------------------+------------------+------------+
| ingestion_timestamp|vertical_category|      avg_baro_alt|      avg_velocity|flying_count|
+--------------------+-----------------+------------------+------------------+------------+
|2026-08-19 14:41:...|       Descending|3725.984638703175

In [131]:
# df.groupBy(col('origin_country')).count().show()
# df.where(col('on_ground')==True ).groupBy(col('origin_country')).count().show()
# df.where(col('on_ground')==False ).groupBy(col('origin_country')).avg('velocity').show()
# df.where(col('on_ground')==False ).groupBy(col('origin_country'),col('vertical_category')).count().orderBy([col('origin_country'),col('vertical_category')],ascending=False).show()
# df.where(col('on_ground')==False ).groupBy(col('origin_country'),col('vertical_category')).avg('velocity').orderBy([col('origin_country'),col('vertical_category')],ascending=False).show()
# df.where(col('on_ground')==False ).groupBy(col('origin_country')).avg('baro_altitude').show()

In [132]:
df_fact=spark.read.format('delta').load("s3a://meddalion/gold/fact_table")
df_dim=spark.read.format('delta').load("s3a://meddalion/gold/dim_table")
df_kpi_ts=spark.read.format('delta').load("s3a://meddalion/gold/KPI_for_timestamps")
df_kpi_cat_ts=spark.read.format('delta').load("s3a://meddalion/gold/KPI_for_cat&ts")

In [134]:
df_kpi_ts.show()

+--------------------+---------------------+------------+---------------+------------------+-----------------+
| ingestion_timestamp|all_observation_count|count_flying|on_ground_count|      avg_velocity|     avg_baro_alt|
+--------------------+---------------------+------------+---------------+------------------+-----------------+
|2026-08-17 22:27:...|                  352|         348|              4| 234.1506905788663|8236.957219967897|
|2026-08-19 14:41:...|                  100|          96|              4|155.72843754291534|5956.141229947408|
+--------------------+---------------------+------------+---------------+------------------+-----------------+



In [133]:
df_kpi_cat_ts.show()

+--------------------+-----------------+------------------+------------------+------------+
| ingestion_timestamp|vertical_category|      avg_baro_alt|      avg_velocity|flying_count|
+--------------------+-----------------+------------------+------------------+------------+
|2026-08-19 14:41:...|       Descending|3725.9846387031753|126.66307693872696|          39|
|2026-08-19 14:41:...|Constant Altitude| 10819.66248936807|229.91000021657635|          31|
|2026-08-19 14:41:...|         Climbing|3502.5623075045073|110.87923064598671|          26|
|2026-08-17 22:27:...|       Descending| 6639.709402645335| 168.3570584584685|         136|
|2026-08-17 22:27:...|Constant Altitude|10613.843558175224|334.85928862435475|         112|
|2026-08-17 22:27:...|         Climbing| 7747.101552734375|210.83640045166015|         100|
+--------------------+-----------------+------------------+------------------+------------+



In [145]:
df=spark.read.format('delta').load("s3a://meddalion/silver/aircraft").where((col('isPoland')==True) & (col('position_source_name')=='ADS-B'))
df_hist=spark.read.format('delta').load("s3a://meddalion/silver/aircraft_hist").where((col('isPoland')==True) & (col('position_source_name')=='ADS-B'))

df_fact=df.select('callsign','last_contact_h','time_position_h','longitude','latitude','geo_altitude','baro_altitude','altitude_diff','true_track','velocity','vertical_category','on_ground','squawk', 'spi')
df_aircraft_dim=df.select('icao24','origin_country','aircraft_category')

# historical data
df_all=df_hist.groupBy(col('ingestion_timestamp')).count().orderBy(col('count'),ascending=False).select('ingestion_timestamp',col('count').alias('all_observation_count'))
df_ground=df_hist.where(col('on_ground')==True ).groupBy(col('ingestion_timestamp')).count().orderBy([col('ingestion_timestamp')],ascending=False).select('ingestion_timestamp',col('count').alias('on_ground_count'))
df_velocity=df_hist.where(col('on_ground')==False).groupBy(col('ingestion_timestamp')).agg(
    avg('velocity').alias('avg_velocity'),
    avg('baro_altitude').alias('avg_baro_alt'),
    count('*').alias('count_flying')
    ).orderBy(col('ingestion_timestamp'),ascending=False).select('ingestion_timestamp','avg_velocity','avg_baro_alt','count_flying')
##################
df_all=df_all.alias('da')\
.join(df_ground.alias('dg'),on=["ingestion_timestamp"],how="left")\
.join(df_velocity.alias('dv'),on=["ingestion_timestamp"],how="left")\
.select('ingestion_timestamp','all_observation_count','count_flying','on_ground_count','dv.avg_velocity','dv.avg_baro_alt')
##################
df_cat=df_hist.where(col('on_ground')==False)\
.groupBy(col('ingestion_timestamp'),col('vertical_category'))\
.agg(
    avg('baro_altitude').alias('avg_baro_alt'),
    avg('velocity').alias('avg_velocity'),
    count('*').alias('flying_count')
    )\
.orderBy([col('ingestion_timestamp'),col('vertical_category')],ascending=False)

In [147]:
df_cat.show()

+--------------------+-----------------+------------------+------------------+------------+
| ingestion_timestamp|vertical_category|      avg_baro_alt|      avg_velocity|flying_count|
+--------------------+-----------------+------------------+------------------+------------+
|2026-08-24 19:21:...|       Descending| 2566.554517139088|109.08022731000727|          44|
|2026-08-24 19:21:...|Constant Altitude| 8692.197469075521|209.19750213623047|          24|
|2026-08-24 19:21:...|         Climbing| 3656.511434500558|127.45857140677316|          35|
|2026-08-24 19:16:...|       Descending|2087.2704166412354| 97.99540008544922|         100|
|2026-08-24 19:16:...|Constant Altitude| 7721.600048700969|195.27041673660278|          48|
|2026-08-24 19:16:...|         Climbing| 4745.227987162272|142.93200047810873|          60|
+--------------------+-----------------+------------------+------------------+------------+



In [28]:
df3 = spark.read.parquet("s3a://meddalion/silver/aircraft_hist").orderBy("ingestion_timestamp",ascending=True).show()

+------+--------+--------------+-------------+------------+---------+--------+------------+---------+--------+----------+-------------+-------+-------------+------+-----+---------------+--------+--------------------+-------------+-------------------+-------------------+-----------------+-----------------+--------------------+--------+
|icao24|callsign|origin_country|time_position|last_contact|longitude|latitude|geo_altitude|on_ground|velocity|true_track|vertical_rate|sensors|baro_altitude|squawk|  spi|position_source|category| ingestion_timestamp|altitude_diff|     last_contact_h|    time_position_h|vertical_category|aircraft_category|position_source_name|isPoland|
+------+--------+--------------+-------------+------------+---------+--------+------------+---------+--------+----------+-------------+-------+-------------+------+-----+---------------+--------+--------------------+-------------+-------------------+-------------------+-----------------+-----------------+--------------------

In [30]:
df3 = spark.read.parquet("s3a://meddalion/gold/KPI_for_timestamps").show()#.orderBy("last_contact_h", ascending=False).show()

+--------------------+---------------------+------------+---------------+------------------+-----------------+
| ingestion_timestamp|all_observation_count|count_flying|on_ground_count|      avg_velocity|     avg_baro_alt|
+--------------------+---------------------+------------+---------------+------------------+-----------------+
|2026-08-28 01:01:...|                   26|          23|              3|166.17434775311014|5470.166037186333|
|2026-08-27 23:44:...|                  110|         106|              4|237.58283093515433|7512.601099554098|
|2026-08-24 19:32:...|                 1116|        1056|             60|133.06920450383967|4700.414282018488|
|2026-08-26 23:18:...|                  760|         710|             50|193.26985841066065|7150.672399708922|
|2026-08-24 19:16:...|                  208|         208|           NULL|133.40596173359796|4154.218861689935|
|2026-08-24 19:21:...|                  106|         103|              3|138.65359273929042| 4364.26250698497|
+

In [3]:
df3 = spark.read.parquet("s3a://meddalion/gold/KPI_for_cat&ts").show()

+--------------------+-----------------+------------------+------------------+------------+
| ingestion_timestamp|vertical_category|      avg_baro_alt|      avg_velocity|flying_count|
+--------------------+-----------------+------------------+------------------+------------+
|2026-08-26 23:18:...|       Descending| 4597.563892733666| 103.0467737490131|         310|
|2026-08-26 23:18:...|Constant Altitude| 10537.24076171875|308.85679809570314|         250|
|2026-08-26 23:18:...|         Climbing| 6782.816044108073|187.08600056966145|         150|
|2026-08-24 19:32:...|       Descending|1957.0065155029297| 88.40174989700317|         480|
|2026-08-24 19:32:...|Constant Altitude| 8061.613555908203|178.17727210304955|         264|
|2026-08-24 19:32:...|         Climbing|  6076.94992182805|163.62000054579514|         312|
|2026-08-24 19:21:...|       Descending| 2566.554517139088|109.08022731000727|          44|
|2026-08-24 19:21:...|Constant Altitude| 8692.197469075521|209.19750213623047|  

In [6]:
df3 = spark.read.parquet("s3a://meddalion/gold/KPI_for_timestamps").show()

+--------------------+---------------------+------------+---------------+------------------+-----------------+
| ingestion_timestamp|all_observation_count|count_flying|on_ground_count|      avg_velocity|     avg_baro_alt|
+--------------------+---------------------+------------+---------------+------------------+-----------------+
|2026-08-24 19:32:...|                 1116|        1056|             60|133.06920450383967|4700.414282018488|
|2026-08-26 23:18:...|                  760|         710|             50|193.26985841066065|7150.672399708922|
|2026-08-24 19:16:...|                  208|         208|           NULL|133.40596173359796|4154.218861689935|
|2026-08-24 19:21:...|                  106|         103|              3|138.65359273929042| 4364.26250698497|
+--------------------+---------------------+------------+---------------+------------------+-----------------+



In [4]:
df3 = spark.read.parquet("s3a://meddalion/gold/dim_table").show()

+------+--------------+-----------------+
|icao24|origin_country|aircraft_category|
+------+--------------+-----------------+
|4ca849|       Ireland|          No Info|
|4bcde8|        Turkey|          No Info|
|4ca741|       Ireland|          No Info|
|48eaf9|        Poland|          No Info|
|48c9e5|        Poland|          No Info|
|48ea23|        Poland|          No Info|
|48ea52|        Poland|          No Info|
|4ca700|       Ireland|          No Info|
|48fcee|        Poland|          No Info|
|48b5c2|        Poland|          No Info|
|49d04d|Czech Republic|          No Info|
|48e979|        Poland|          No Info|
|48e94b|        Poland|          No Info|
|48e951|        Poland|          No Info|
|48e93d|        Poland|          No Info|
|48e93b|        Poland|          No Info|
|48c1a6|        Poland|          No Info|
|4ca14e|       Ireland|          No Info|
|48b111|        Poland|          No Info|
|48b101|        Poland|          No Info|
+------+--------------+-----------